In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver

In [8]:
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [12]:
llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

response = llm.invoke("Where is India? Answer in one sentence.")

print(response.content)

[{'type': 'text', 'text': 'India is located in South Asia, bordered by the Indian Ocean to the south, the Arabian Sea to the southwest, and the Bay of Bengal to the southeast.', 'extras': {'signature': 'EmAKXgFpFH0TdnC8hyYFS4ht6m4AQ3KTgk3kZF56ALnTP8sKW5XmHoij1lbiuUpdQEId/6E8wbM5miJvAMYID1nC5x10cuY1JF+WAeSpMWTE2mvYuODFH9MTa+QJvsHpDhw='}}]


In [13]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [14]:
def chat_node(state: ChatState):
    # take user querey from state
    messages = state['messages']
    # send to llm
    response = llm.invoke(messages)
    # response store state
    return {'messages': [response]}

In [15]:
checkpoint = MemorySaver()
graph = StateGraph(ChatState)

# add node
graph.add_node('chat_node', chat_node)

# add_edge
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

# execute the graph
chatbot = graph.compile(checkpointer=checkpoint)

In [ ]:
chatbot

In [16]:
thread_id = "1"
initial_state = {
    'messages' : [HumanMessage(content='What is india')]
}
config = {'configurable': {'thread_id': thread_id}}
response = chatbot.invoke(initial_state,config=config)

In [17]:
response['messages'][-1].content

[{'type': 'text',
  'text': "**India**, officially known as the **Republic of India** (Hindi: *Bhārat Gaṇarājya*), is a country in South Asia. It is one of the most historically rich, culturally diverse, and populous nations in the world. \n\nHere is a breakdown of what India is, covering its key aspects:\n\n### 1. Geography and Demographics\n* **Location:** India occupies the majority of the Indian subcontinent. It shares land borders with Pakistan, China, Nepal, Bhutan, Myanmar, and Bangladesh, and maritime borders with Sri Lanka, Maldives, and Indonesia.\n* **Capital:** New Delhi.\n* **Population:** India is the **most populous country in the world** (surpassing China in 2023), with over 1.4 billion people.\n* **Languages:** India has no single national language, but Hindi is the official language of the central government, alongside English. The Indian Constitution recognizes **22 major languages**, and hundreds of dialects are spoken across the country.\n\n### 2. Government and Po

In [18]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='What is india', additional_kwargs={}, response_metadata={}, id='b48e0c0b-3642-4780-ae11-e53b4dfb58fd'), AIMessage(content=[{'type': 'text', 'text': "**India**, officially known as the **Republic of India** (Hindi: *Bhārat Gaṇarājya*), is a country in South Asia. It is one of the most historically rich, culturally diverse, and populous nations in the world. \n\nHere is a breakdown of what India is, covering its key aspects:\n\n### 1. Geography and Demographics\n* **Location:** India occupies the majority of the Indian subcontinent. It shares land borders with Pakistan, China, Nepal, Bhutan, Myanmar, and Bangladesh, and maritime borders with Sri Lanka, Maldives, and Indonesia.\n* **Capital:** New Delhi.\n* **Population:** India is the **most populous country in the world** (surpassing China in 2023), with over 1.4 billion people.\n* **Languages:** India has no single national language, but Hindi is the official language of the cent

In [19]:
while True:
    user_message = input('Type here: ')

    print('User: ', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    # this is called persistence

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]},config = config)
    print('AI: ',response['messages'][-1].content)

User:  exit
